# Exploratory Data Analysis: Contactless Palmprint Dataset

This notebook explores the Tongji contactless palmprint dataset before preprocessing and model training. The raw data has two flat folders, `session1` and `session2`. Each group of ten consecutive filenames belongs to one palm identity: `00001`–`00010` are palm 1, `00011`–`00020` are palm 2, and so on.


In [ ]:
# Cell 1: Import libraries
import os
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

SEED = 42
IMAGE_EXTENSIONS = {'.tif', '.tiff', '.png', '.jpg', '.jpeg'}
print('Libraries imported successfully.')


In [ ]:
# Cell 2: Find and explore the dataset
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

configured_root = os.environ.get('PALM_DATA_ROOT')
candidates = [Path(configured_root).expanduser() if configured_root else None]
candidates += [
    PROJECT_ROOT / 'data' / 'raw' / 'palm-dataset',
    PROJECT_ROOT / 'data' / 'raw',
    PROJECT_ROOT / 'palm-dataset',
]
DATA_ROOT = next(
    (candidate.resolve() for candidate in candidates
     if candidate is not None
     and (candidate / 'session1').is_dir()
     and (candidate / 'session2').is_dir()),
    None,
)

if DATA_ROOT is None:
    import kagglehub
    DATA_ROOT = Path(kagglehub.dataset_download('saqibshoaibdz/palm-dataset'))

print(f'Dataset root: {DATA_ROOT}')
for session in ('session1', 'session2'):
    count = sum(path.suffix.lower() in IMAGE_EXTENSIONS for path in (DATA_ROOT / session).iterdir())
    print(f'{session}: {count:,} image files')


In [ ]:
# Cell 3: Build a table of images and palm identities
def collect_images(session):
    rows = []
    for image_path in sorted((DATA_ROOT / session).iterdir()):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        try:
            image_number = int(image_path.stem)
        except ValueError:
            continue
        # Ten consecutive files belong to the same palm identity.
        palm_id = (image_number - 1) // 10 + 1
        rows.append({
            'session': session,
            'filename': image_path.name,
            'path': image_path,
            'image_number': image_number,
            'palm_id': palm_id,
        })
    return rows

images_df = pd.DataFrame(collect_images('session1') + collect_images('session2'))
images_per_session = images_df.groupby('session').size()
images_per_palm = images_df.groupby(['session', 'palm_id']).size()

print(f'Total images: {len(images_df):,}')
print(f'Number of palm identities: {images_df.palm_id.nunique():,}')
print(f'Images per palm per session: {sorted(images_per_palm.unique())}')
display(images_df.head())


In [ ]:
# Cell 4: Inspect image dimensions from a reproducible sample
sample_for_dimensions = images_df.sample(n=min(50, len(images_df)), random_state=SEED)
dimensions = []

for row in sample_for_dimensions.itertuples(index=False):
    with Image.open(row.path) as image:
        dimensions.append({
            'session': row.session,
            'filename': row.filename,
            'width': image.width,
            'height': image.height,
            'mode': image.mode,
        })

dimensions_df = pd.DataFrame(dimensions)
display(dimensions_df.head(10))
print('Most common dimensions:', Counter(zip(dimensions_df.width, dimensions_df.height)).most_common(1)[0])


In [ ]:
# Cell 5: Display sample raw palm images
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sample_images = images_df[images_df.session == 'session1'].sample(n=9, random_state=SEED)
fig, axes = plt.subplots(3, 3, figsize=(12, 10))

for axis, row in zip(axes.flat, sample_images.itertuples(index=False)):
    with Image.open(row.path) as image:
        axis.imshow(image)
    axis.set_title(f'Palm {row.palm_id:03d} | {row.filename}')
    axis.axis('off')

plt.suptitle('Sample Raw Palm Images: Session 1', y=0.92)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'eda_sample_palms.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 6: Image and dataset statistics
print('Image dimension statistics from the 50-image sample:')
display(dimensions_df[['width', 'height']].describe().round(2))

print('\nDataset statistics:')
print(f'- Total images: {len(images_df):,}')
print(f'- Session 1 images: {images_per_session.get("session1", 0):,}')
print(f'- Session 2 images: {images_per_session.get("session2", 0):,}')
print(f'- Palm identities: {images_df.palm_id.nunique():,}')
print(f'- Average images per palm across both sessions: {len(images_df) / images_df.palm_id.nunique():.2f}')
print(f'- Image modes in sample: {dimensions_df.mode.value_counts().to_dict()}')


In [ ]:
# Cell 7: Distribution of images across palm identities
counts_by_palm = images_df.groupby(['palm_id', 'session']).size().unstack(fill_value=0)

plt.figure(figsize=(16, 5))
plt.plot(counts_by_palm.index, counts_by_palm.get('session1', 0), label='Session 1', linewidth=1)
plt.plot(counts_by_palm.index, counts_by_palm.get('session2', 0), label='Session 2', linewidth=1)
plt.xlabel('Palm identity')
plt.ylabel('Number of images')
plt.title('Distribution of Palm Images Across Identities')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'eda_palm_image_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Images per palm per session:')
display(counts_by_palm.describe().round(2))
print('EDA complete.')
